## Section 7 — Grad-CAM : Explainabilite de l'IA (XAI)

### Pourquoi l'Explainabilite est cruciale en medical ?

Un modele qui predit "PNEUMONIA" sans expliquer pourquoi n'est pas
exploitable cliniquement. Le medecin a besoin de savoir **ou le modele
a regarde** pour prendre sa decision.

**Grad-CAM (Gradient-weighted Class Activation Mapping)** :
calcule le gradient de la prediction par rapport aux feature maps
de la derniere couche convolutive. Les zones avec un gradient eleve
sont celles qui ont le plus influence la prediction.

```
Resultat : carte de chaleur superposee sur la radio
Rouge  = zone tres influente pour la prediction
Bleu   = zone peu influente
```

### Utilite clinique
- Verifier que le modele regarde les bonnes zones (infiltrats, opacites)
- Detecter si le modele fait des predictions pour de mauvaises raisons
- Renforcer la confiance des medecins dans les predictions

In [ ]:
# ============================================================
# SECTION 7 : GRAD-CAM
# ============================================================

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
import numpy as np

def trouver_derniere_couche_conv(model, nom):
    """
    Identifie la derniere couche convolutive.
    Pour les modeles geles, on active temporairement les gradients.
    """
    if nom == "M1 CNN Baseline":
        return [model.conv3[0]]

    elif nom == "M2 ResNet18":
        return [model.layer4[-1].conv2]

    elif nom == "M3 DenseNet-121":
        # Chercher la derniere _DenseLayer
        for name, module in reversed(list(model.features.named_modules())):
            if isinstance(module, nn.Conv2d):
                return [module]

    elif nom == "M4 EfficientNet-V2":
        for name, module in reversed(list(model.features.named_modules())):
            if isinstance(module, nn.Conv2d):
                return [module]

    elif nom == "M5 CNN+ViT":
        # Backbone est un Sequential de blocs ResNet
        # Le dernier bloc est backbone[-1] qui est un Sequential
        # La derniere conv est conv2 du dernier BasicBlock
        last_block = list(model.backbone.children())[-1]
        if hasattr(last_block, '__iter__'):
            for sub in reversed(list(last_block.children())):
                if isinstance(sub, nn.Conv2d):
                    return [sub]
        return [list(model.backbone.modules())[-1]]

    return None

def activer_gradients_pour_gradcam(model, nom):
    """
    Active les gradients sur la couche cible uniquement.
    Necessaire car les couches pre-entrainement sont gelees
    (requires_grad=False), ce qui cause l'erreur NoneType.
    """
    target_layers = trouver_derniere_couche_conv(model, nom)
    if target_layers and target_layers[0] is not None:
        for param in target_layers[0].parameters():
            param.requires_grad = True
    return target_layers

def visualiser_gradcam_v2(model, test_loader, nom_modele, nb_images=8):
    """
    Grad-CAM avec activation forcee des gradients sur la couche cible.
    """
    model.eval()

    # Activer les gradients sur la couche cible
    target_layers = activer_gradients_pour_gradcam(model, nom_modele)

    if target_layers is None or target_layers[0] is None:
        print(f"Couche introuvable pour {nom_modele}")
        return

    try:
        # use_cuda=False car on est sur CPU
        cam = GradCAM(model=model, target_layers=target_layers)
    except Exception as e:
        print(f"Erreur init GradCAM {nom_modele} : {e}")
        return

    # Collecter exemples
    exemples_n, exemples_p = [], []
    for imgs, labels in test_loader:
        for img, label in zip(imgs, labels):
            if label.item() == 0 and len(exemples_n) < nb_images//2:
                exemples_n.append((img, 0))
            if label.item() == 1 and len(exemples_p) < nb_images//2:
                exemples_p.append((img, 1))
        if len(exemples_n) >= nb_images//2 and len(exemples_p) >= nb_images//2:
            break

    tous = exemples_n + exemples_p
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])

    fig, axes = plt.subplots(3, nb_images, figsize=(nb_images*2.5, 9))
    fig.suptitle(
        f"Grad-CAM — {nom_modele}\n"
        f"Ligne 1 : Originale | Ligne 2 : Carte thermique | Ligne 3 : Superposition",
        fontsize=11, fontweight="bold"
    )

    for i, (img_tensor, true_label) in enumerate(tous[:nb_images]):
        # IMPORTANT : requires_grad=True sur l'input pour Grad-CAM
        input_tensor = img_tensor.unsqueeze(0).to(device)
        input_tensor.requires_grad = True

        with torch.enable_grad():   # activer les gradients meme en eval()
            out       = model(input_tensor)
            probs     = torch.softmax(out, dim=1)[0]
            pred      = out.argmax(dim=1).item()
            prob_pred = probs[pred].item()

        try:
            grayscale_cam = cam(
                input_tensor=input_tensor,
                targets=[ClassifierOutputTarget(pred)]
            )[0]
        except Exception as e:
            print(f"Erreur image {i} : {e}")
            continue

        # Denormaliser
        img_np   = img_tensor.permute(1,2,0).numpy()
        img_np   = (img_np * std + mean).clip(0, 1)
        img_gray = img_np[:,:,0]
        img_rgb  = np.stack([img_gray]*3, axis=-1)
        cam_img  = show_cam_on_image(
            img_rgb.astype(np.float32),
            grayscale_cam, use_rgb=True
        )

        c_vraie = "steelblue" if true_label == 0 else "tomato"
        c_pred  = "steelblue" if pred == 0 else "tomato"
        l_vraie = "NORMAL" if true_label == 0 else "PNEUMONIE"
        l_pred  = "NORMAL" if pred == 0 else "PNEUMONIE"
        correct = "✓" if true_label == pred else "✗"

        axes[0][i].imshow(img_gray, cmap="gray")
        axes[0][i].set_title(f"Vraie:\n{l_vraie}",
                              color=c_vraie, fontsize=8, fontweight="bold")
        axes[0][i].axis("off")

        axes[1][i].imshow(grayscale_cam, cmap="jet")
        axes[1][i].set_title("Carte\nGrad-CAM", fontsize=8)
        axes[1][i].axis("off")

        axes[2][i].imshow(cam_img)
        axes[2][i].set_title(f"Pred: {l_pred}\n{prob_pred*100:.0f}% {correct}",
                              color=c_pred, fontsize=8, fontweight="bold")
        axes[2][i].axis("off")

    axes[0][0].set_ylabel("Original",     fontsize=9, rotation=90, labelpad=8)
    axes[1][0].set_ylabel("Grad-CAM",     fontsize=9, rotation=90, labelpad=8)
    axes[2][0].set_ylabel("Superposition",fontsize=9, rotation=90, labelpad=8)

    plt.tight_layout()
    plt.show()

# ── Appliquer sur les 5 modeles ───────────────────────────────
print("Grad-CAM sur les 5 modeles...")
for model_ref, nom in [
    (model1, "M1 CNN Baseline"),
    (model2, "M2 ResNet18"),
    (model3, "M3 DenseNet-121"),
    (model4, "M4 EfficientNet-V2"),
    (model5, "M5 CNN+ViT")
]:
    print(f"\n--- {nom} ---")
    try:
        visualiser_gradcam_v2(model_ref, test_loader, nom, nb_images=8)
    except Exception as e:
        print(f"Erreur {nom} : {e}")